In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean
from sklearn.naive_bayes import MultinomialNB

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
data = pd.read_excel('dataset_gen.xlsx') 
dataset_all_gen = pd.read_excel('dataset_all_gen.xlsx')
dataset_all_original = pd.read_excel('dataset_all_original.xlsx')
dataset_jaccard_gen = pd.read_excel('dataset_jaccard_gen.xlsx')
dataset_jaccard_original = pd.read_excel('dataset_jaccard_original.xlsx')
dataset_myroom_gen = pd.read_excel('dataset_myroom_gen.xlsx')
dataset_myroom_original = pd.read_excel('dataset_myroom_original.xlsx')
dataset_university_gen = pd.read_excel('dataset_university_gen.xlsx')
dataset_university_original = pd.read_excel('dataset_university_original.xlsx')

In [4]:
data = data.sample(frac = 1) 
dataset_all_gen = dataset_all_gen.sample(frac = 1) 
dataset_all_original = dataset_all_original.sample(frac = 1) 
dataset_jaccard_gen = dataset_jaccard_gen.sample(frac = 1) 
dataset_jaccard_original = dataset_jaccard_original.sample(frac = 1) 
dataset_myroom_gen = dataset_myroom_gen.sample(frac = 1) 
dataset_myroom_original = dataset_myroom_original.sample(frac = 1) 
dataset_university_gen = dataset_university_gen.sample(frac = 1) 
dataset_university_original = dataset_university_original.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)
dataset_all_gen = dataset_all_gen.reset_index(drop=True)
dataset_all_original = dataset_all_original.reset_index(drop=True)
dataset_jaccard_gen = dataset_jaccard_gen.reset_index(drop=True)
dataset_jaccard_original = dataset_jaccard_original.reset_index(drop=True)
dataset_myroom_gen = dataset_myroom_gen.reset_index(drop=True)
dataset_myroom_original = dataset_myroom_original.reset_index(drop=True)
dataset_university_gen = dataset_university_gen.reset_index(drop=True)
dataset_university_original = dataset_university_original.reset_index(drop=True)

In [6]:
data.isnull().sum()

type             0
Name             0
Vendor           0
DNS           2304
mDNS          2704
NUPNP        11000
open_port     1731
SSDP_Type     2304
SSDP_t       11000
SSDP_v       11000
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
#fill Na
dataset_all_gen = dataset_all_gen.fillna("NaN")
dataset_all_original = dataset_all_original.fillna("NaN")
dataset_jaccard_gen = dataset_jaccard_gen.fillna("NaN")
dataset_jaccard_original = dataset_jaccard_original.fillna("NaN")
dataset_myroom_gen = dataset_myroom_gen.fillna("NaN")
dataset_myroom_original = dataset_myroom_original.fillna("NaN")
dataset_university_gen = dataset_university_gen.fillna("NaN")
dataset_university_original = dataset_university_original.fillna("NaN")

In [9]:
data.isnull().sum()

type         0
Name         0
Vendor       0
DNS          0
mDNS         0
NUPNP        0
open_port    0
SSDP_Type    0
SSDP_t       0
SSDP_v       0
dtype: int64

In [10]:
data.dtypes

type         object
Name         object
Vendor       object
DNS          object
mDNS         object
NUPNP        object
open_port    object
SSDP_Type    object
SSDP_t       object
SSDP_v       object
dtype: object

In [11]:
data = data.astype('category')

In [12]:
dataset_all_gen = dataset_all_gen.astype('category')
dataset_all_original = dataset_all_original.astype('category')
dataset_jaccard_gen = dataset_jaccard_gen.astype('category')
dataset_jaccard_original = dataset_jaccard_original.astype('category')
dataset_myroom_gen = dataset_myroom_gen.astype('category')
dataset_myroom_original = dataset_myroom_original.astype('category')
dataset_university_gen = dataset_university_gen.astype('category')
dataset_university_original = dataset_university_original.astype('category')

In [13]:
data.dtypes

type         category
Name         category
Vendor       category
DNS          category
mDNS         category
NUPNP        category
open_port    category
SSDP_Type    category
SSDP_t       category
SSDP_v       category
dtype: object

In [14]:
data

,type,Name,Vendor,DNS,mDNS,NUPNP,open_port,SSDP_Type,SSDP_t,SSDP_v
0,Non-IOT,Lenovo Win10 laptop,Lenovo,t.co,_gembucket._tcp.local.,NaN,tcp/80,M-SEARCH * HTTP/1.1,NaN,NaN
1,Non-IOT,Lg Smartphone,LG Electronics,hud.gov,_fixflex._udp.local.,NaN,tcp/80,M-SEARCH * HTTP/1.1,NaN,NaN
2,Non-IOT,Apple Macbook,"Apple, Inc.",alibaba.com,NaN,NaN,NaN,M-SEARCH * HTTP/1.1,NaN,NaN
3,Cameras,Nest Dropcam,Nest Labs Inc.,nytimes.com,_alpha._udp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NaN,NaN
4,Non-IOT,Mac laptop,"Apple, Inc.",NaN,_tin._tcp.local.,NaN,tcp/80,NaN,NaN,NaN
5,Non-IOT,MacBook/Iphone,"Apple, Inc.",java.com,NaN,NaN,tcp/22,M-SEARCH * HTTP/1.1,NaN,NaN
6,Health-Monitor,Withings Smart Monitor,Withings,zdnet.com,_voyatouch._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
7,Non-IOT,Samsung Galaxy Tab,SAMSUNG,NaN,NaN,NaN,tcp/22,NaN,NaN,NaN
8,Cameras,NestCam,Nest Labs Inc.,qq.com,NaN,NaN,tcp/80,NOTIFY * HTTP/1.1,NaN,NaN
9,Energy Management,Koogeek Lightbulb,Koogeek Inc,storify.com,_namfix._tcp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NaN,NaN


In [15]:
y = data.drop_duplicates(keep='first')
y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [16]:
len(y)

10610

In [17]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original.drop_duplicates(keep='first')

In [18]:
#drop name
dataset_all_gen_y = dataset_all_gen_y.drop(['Name'], axis=1)
dataset_all_original_y = dataset_all_original_y.drop(['Name'], axis=1)
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop(['Name'], axis=1)
dataset_jaccard_original_y = dataset_jaccard_original_y.drop(['Name'], axis=1)
dataset_myroom_gen_y = dataset_myroom_gen_y.drop(['Name'], axis=1)
dataset_myroom_original_y = dataset_myroom_original_y.drop(['Name'], axis=1)
dataset_university_gen_y = dataset_university_gen_y.drop(['Name'], axis=1)
dataset_university_original_y = dataset_university_original_y.drop(['Name'], axis=1)

In [19]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen_y.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original_y.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original_y.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen_y.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original_y.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen_y.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original_y.drop_duplicates(keep='first')

In [20]:
y = y.apply(LabelEncoder().fit_transform)

In [21]:
dataset_all_gen_y = dataset_all_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_all_original_y = dataset_all_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_original_y = dataset_jaccard_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_gen_y = dataset_myroom_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_original_y = dataset_myroom_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_gen_y = dataset_university_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_original_y = dataset_university_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

In [22]:
list_data = []
list_data.append(y)
list_data.append(dataset_all_gen_y)
list_data.append(dataset_all_original_y)
list_data.append(dataset_jaccard_gen_y)
list_data.append(dataset_jaccard_original_y)
list_data.append(dataset_myroom_gen_y)
list_data.append(dataset_myroom_original_y)
list_data.append(dataset_university_gen_y)
list_data.append(dataset_university_original_y)
len(list_data)

9

In [23]:
accuracy = []
accuracy_t = []
accuracy_temp = []

In [24]:
for data in list_data:
    for i in range(100):
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        for train_index, test_index in kf.split(data):
        #     print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
            clf = MultinomialNB()
#             clf = tree.DecisionTreeClassifier()
#             clf = clf.fit(train_data,label_data )
            clf = clf.fit(train_data,label_data )
#             predict = clf.predict(test_data)
            predict = clf.predict(test_data)
#             print(truelabel_data)
#             print(predict)
        #     len(predict)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_t))

0   Accuracy:  0.24410933081998115
0   Accuracy:  0.22808671065032987
0   Accuracy:  0.23374175306314798
0   Accuracy:  0.21489161168708765
0   Accuracy:  0.27238454288407166
0   Accuracy:  0.2478793590951932
0   Accuracy:  0.21583411875589067
0   Accuracy:  0.26013195098963243
0   Accuracy:  0.23279924599434496
0   Accuracy:  0.2111215834118756
1   Accuracy:  0.19698397737983034
1   Accuracy:  0.22808671065032987
1   Accuracy:  0.2582469368520264
1   Accuracy:  0.23468426013195098
1   Accuracy:  0.20452403393025448
1   Accuracy:  0.2233741753063148
1   Accuracy:  0.28086710650329877
1   Accuracy:  0.2554194156456173
1   Accuracy:  0.23845428840716307
1   Accuracy:  0.22148916116870876
2   Accuracy:  0.22808671065032987
2   Accuracy:  0.2469368520263902
2   Accuracy:  0.24222431668237512
2   Accuracy:  0.26013195098963243
2   Accuracy:  0.236569274269557
2   Accuracy:  0.21489161168708765
2   Accuracy:  0.21206409048067862
2   Accuracy:  0.21866163996229973
2   Accuracy:  0.26201696512

23   Accuracy:  0.22148916116870876
23   Accuracy:  0.24505183788878418
23   Accuracy:  0.22148916116870876
23   Accuracy:  0.26578699340245054
24   Accuracy:  0.2573044297832234
24   Accuracy:  0.2233741753063148
24   Accuracy:  0.21583411875589067
24   Accuracy:  0.22054665409990576
24   Accuracy:  0.235626767200754
24   Accuracy:  0.23939679547596607
24   Accuracy:  0.22620169651272384
24   Accuracy:  0.2525918944392083
24   Accuracy:  0.23374175306314798
24   Accuracy:  0.22714420358152687
25   Accuracy:  0.21394910461828465
25   Accuracy:  0.24316682375117812
25   Accuracy:  0.2412818096135721
25   Accuracy:  0.21677662582469368
25   Accuracy:  0.2478793590951932
25   Accuracy:  0.2648444863336475
25   Accuracy:  0.21583411875589067
25   Accuracy:  0.21394910461828465
25   Accuracy:  0.2573044297832234
25   Accuracy:  0.22054665409990576
26   Accuracy:  0.26107445805843543
26   Accuracy:  0.24882186616399624
26   Accuracy:  0.22714420358152687
26   Accuracy:  0.23939679547596607
2

47   Accuracy:  0.24410933081998115
47   Accuracy:  0.22054665409990576
47   Accuracy:  0.2299717247879359
47   Accuracy:  0.20923656927426956
47   Accuracy:  0.26013195098963243
47   Accuracy:  0.23374175306314798
48   Accuracy:  0.23374175306314798
48   Accuracy:  0.2544769085768143
48   Accuracy:  0.21677662582469368
48   Accuracy:  0.22525918944392084
48   Accuracy:  0.23751178133836004
48   Accuracy:  0.19604147031102734
48   Accuracy:  0.2403393025447691
48   Accuracy:  0.2478793590951932
48   Accuracy:  0.23939679547596607
48   Accuracy:  0.2695570216776626
49   Accuracy:  0.26013195098963243
49   Accuracy:  0.21489161168708765
49   Accuracy:  0.2478793590951932
49   Accuracy:  0.22714420358152687
49   Accuracy:  0.23279924599434496
49   Accuracy:  0.22808671065032987
49   Accuracy:  0.2412818096135721
49   Accuracy:  0.21583411875589067
49   Accuracy:  0.2535344015080113
49   Accuracy:  0.2573044297832234
50   Accuracy:  0.26861451460885954
50   Accuracy:  0.21866163996229973
5

70   Accuracy:  0.2544769085768143
70   Accuracy:  0.19038642789820923
71   Accuracy:  0.27521206409048066
71   Accuracy:  0.21394910461828465
71   Accuracy:  0.21583411875589067
71   Accuracy:  0.2299717247879359
71   Accuracy:  0.2233741753063148
71   Accuracy:  0.23374175306314798
71   Accuracy:  0.21583411875589067
71   Accuracy:  0.2299717247879359
71   Accuracy:  0.2695570216776626
71   Accuracy:  0.2478793590951932
72   Accuracy:  0.2629594721960415
72   Accuracy:  0.236569274269557
72   Accuracy:  0.19604147031102734
72   Accuracy:  0.24599434495758718
72   Accuracy:  0.22148916116870876
72   Accuracy:  0.22148916116870876
72   Accuracy:  0.21017907634307256
72   Accuracy:  0.236569274269557
72   Accuracy:  0.26013195098963243
72   Accuracy:  0.2554194156456173
73   Accuracy:  0.24599434495758718
73   Accuracy:  0.22431668237511782
73   Accuracy:  0.19132893496701225
73   Accuracy:  0.2525918944392083
73   Accuracy:  0.24222431668237512
73   Accuracy:  0.23185673892554196
73   

94   Accuracy:  0.2412818096135721
94   Accuracy:  0.22148916116870876
94   Accuracy:  0.23845428840716307
94   Accuracy:  0.21677662582469368
94   Accuracy:  0.21677662582469368
94   Accuracy:  0.2469368520263902
94   Accuracy:  0.2573044297832234
94   Accuracy:  0.2573044297832234
94   Accuracy:  0.2554194156456173
95   Accuracy:  0.2478793590951932
95   Accuracy:  0.2111215834118756
95   Accuracy:  0.23751178133836004
95   Accuracy:  0.23374175306314798
95   Accuracy:  0.20358152686145145
95   Accuracy:  0.21394910461828465
95   Accuracy:  0.26578699340245054
95   Accuracy:  0.2478793590951932
95   Accuracy:  0.2554194156456173
95   Accuracy:  0.24599434495758718
96   Accuracy:  0.21960414703110273
96   Accuracy:  0.25070688030160226
96   Accuracy:  0.19698397737983034
96   Accuracy:  0.21960414703110273
96   Accuracy:  0.21866163996229973
96   Accuracy:  0.27426955702167766
96   Accuracy:  0.2412818096135721
96   Accuracy:  0.24410933081998115
96   Accuracy:  0.2177191328934967
96 

17   Accuracy:  0.25
17   Accuracy:  0.24822695035460993
18   Accuracy:  0.23472099202834368
18   Accuracy:  0.254207263064659
18   Accuracy:  0.27989371124889284
18   Accuracy:  0.2736935341009743
18   Accuracy:  0.2604074402125775
18   Accuracy:  0.24534986713906112
18   Accuracy:  0.24269264836138174
18   Accuracy:  0.24911347517730498
18   Accuracy:  0.2695035460992908
18   Accuracy:  0.26152482269503546
19   Accuracy:  0.2630646589902569
19   Accuracy:  0.25155004428697963
19   Accuracy:  0.2577502214348981
19   Accuracy:  0.25597874224977857
19   Accuracy:  0.2657218777679362
19   Accuracy:  0.25863596102745795
19   Accuracy:  0.2329495128432241
19   Accuracy:  0.24822695035460993
19   Accuracy:  0.2429078014184397
19   Accuracy:  0.24911347517730498
20   Accuracy:  0.25066430469441986
20   Accuracy:  0.2471213463241807
20   Accuracy:  0.24180690876882197
20   Accuracy:  0.27015057573073514
20   Accuracy:  0.2612931798051373
20   Accuracy:  0.27989371124889284
20   Accuracy:  0.2

... (27 outputs left out)

12   Accuracy:  0.5357142857142857
13   Accuracy:  0.3793103448275862
13   Accuracy:  0.4827586206896552
13   Accuracy:  0.3448275862068966
13   Accuracy:  0.5517241379310345
13   Accuracy:  0.4642857142857143
13   Accuracy:  0.4642857142857143
13   Accuracy:  0.5
13   Accuracy:  0.35714285714285715
13   Accuracy:  0.5
13   Accuracy:  0.6071428571428571
14   Accuracy:  0.4482758620689655
14   Accuracy:  0.3448275862068966
14   Accuracy:  0.3448275862068966
14   Accuracy:  0.6206896551724138
14   Accuracy:  0.6071428571428571
14   Accuracy:  0.35714285714285715
14   Accuracy:  0.42857142857142855
14   Accuracy:  0.5
14   Accuracy:  0.39285714285714285
14   Accuracy:  0.5
15   Accuracy:  0.4827586206896552
15   Accuracy:  0.5517241379310345
15   Accuracy:  0.3103448275862069
15   Accuracy:  0.5172413793103449
15   Accuracy:  0.4642857142857143
15   Accuracy:  0.4642857142857143
15   Accuracy:  0.4642857142857143
15   Accuracy:  0.5
15   Accuracy:  0.4642857142857143
15   Accuracy:  0.535

37   Accuracy:  0.5
38   Accuracy:  0.5517241379310345
38   Accuracy:  0.5517241379310345
38   Accuracy:  0.5172413793103449
38   Accuracy:  0.5172413793103449
38   Accuracy:  0.25
38   Accuracy:  0.35714285714285715
38   Accuracy:  0.39285714285714285
38   Accuracy:  0.5357142857142857
38   Accuracy:  0.32142857142857145
38   Accuracy:  0.5357142857142857
39   Accuracy:  0.4482758620689655
39   Accuracy:  0.27586206896551724
39   Accuracy:  0.3793103448275862
39   Accuracy:  0.6206896551724138
39   Accuracy:  0.5357142857142857
39   Accuracy:  0.39285714285714285
39   Accuracy:  0.42857142857142855
39   Accuracy:  0.42857142857142855
39   Accuracy:  0.4642857142857143
39   Accuracy:  0.5714285714285714
40   Accuracy:  0.41379310344827586
40   Accuracy:  0.4827586206896552
40   Accuracy:  0.3793103448275862
40   Accuracy:  0.41379310344827586
40   Accuracy:  0.4642857142857143
40   Accuracy:  0.4642857142857143
40   Accuracy:  0.42857142857142855
40   Accuracy:  0.39285714285714285
40 

63   Accuracy:  0.5172413793103449
63   Accuracy:  0.4827586206896552
63   Accuracy:  0.41379310344827586
63   Accuracy:  0.4482758620689655
63   Accuracy:  0.4642857142857143
63   Accuracy:  0.5357142857142857
63   Accuracy:  0.4642857142857143
63   Accuracy:  0.2857142857142857
63   Accuracy:  0.39285714285714285
63   Accuracy:  0.5714285714285714
64   Accuracy:  0.3793103448275862
64   Accuracy:  0.4482758620689655
64   Accuracy:  0.5172413793103449
64   Accuracy:  0.5517241379310345
64   Accuracy:  0.35714285714285715
64   Accuracy:  0.6071428571428571
64   Accuracy:  0.39285714285714285
64   Accuracy:  0.5714285714285714
64   Accuracy:  0.39285714285714285
64   Accuracy:  0.42857142857142855
65   Accuracy:  0.5172413793103449
65   Accuracy:  0.3793103448275862
65   Accuracy:  0.5862068965517241
65   Accuracy:  0.3103448275862069
65   Accuracy:  0.6071428571428571
65   Accuracy:  0.35714285714285715
65   Accuracy:  0.5714285714285714
65   Accuracy:  0.42857142857142855
65   Accurac

87   Accuracy:  0.39285714285714285
87   Accuracy:  0.42857142857142855
87   Accuracy:  0.5
87   Accuracy:  0.42857142857142855
87   Accuracy:  0.5357142857142857
87   Accuracy:  0.4642857142857143
88   Accuracy:  0.6206896551724138
88   Accuracy:  0.5862068965517241
88   Accuracy:  0.5172413793103449
88   Accuracy:  0.41379310344827586
88   Accuracy:  0.5357142857142857
88   Accuracy:  0.39285714285714285
88   Accuracy:  0.4642857142857143
88   Accuracy:  0.21428571428571427
88   Accuracy:  0.35714285714285715
88   Accuracy:  0.4642857142857143
89   Accuracy:  0.41379310344827586
89   Accuracy:  0.5517241379310345
89   Accuracy:  0.6551724137931034
89   Accuracy:  0.41379310344827586
89   Accuracy:  0.42857142857142855
89   Accuracy:  0.42857142857142855
89   Accuracy:  0.39285714285714285
89   Accuracy:  0.42857142857142855
89   Accuracy:  0.5
89   Accuracy:  0.39285714285714285
90   Accuracy:  0.41379310344827586
90   Accuracy:  0.5172413793103449
90   Accuracy:  0.4482758620689655


In [25]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[2])
print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[4])
print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[6])
print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[8])

10610 	 Accuracy for dataset_gen: 	 0.23630637978299654
11287 	 Accuracy for dataset_all_gen: 	 0.23946269301461354
677 	 Accuracy for dataset_all_original: 	 0.25960715415216834
10912 	 Accuracy for dataset_jaccard_gen: 	 0.2766279923154014
302 	 Accuracy for dataset_jaccard_original: 	 0.29713394439133406
10701 	 Accuracy for dataset_myroom_gen: 	 0.3163918598796537
91 	 Accuracy for dataset_myroom_original: 	 0.33242325517329047
10894 	 Accuracy for dataset_university_gen: 	 0.3462481410553003
284 	 Accuracy for dataset_university_original: 	 0.35585034191152354
